In [5]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ================= 1. 数据加载与预处理 =================
file_path = '智能健康手环数据集.xlsx'
df = pd.read_excel(file_path)

print(df.head())

# 去除所有列名前后的隐藏空格
df.columns = df.columns.str.strip()

# 【已修复】直接使用你数据集中的真实完整列名
real_time_col = '日期'
real_steps_col = '步数'

print("✅ 成功识别到核心字段，开始分析...")
print("=" * 60)

# ================= 2. 一、用户活动模式分析 =================
print("\n【一、用户活动模式分析】")

# 将日期列转换为 datetime 格式
df[real_time_col] = pd.to_datetime(df[real_time_col])

# 提取小时数（如果数据中包含具体时间）
if df[real_time_col].dt.hour.nunique() > 1:
    # 如果包含具体时分秒，按小时段划分
    df['hour'] = df[real_time_col].dt.hour
    
    # 【Bug修复】7个边界点，必须对应6个标签
    bins = [0, 5, 6, 8, 17, 20, 24]
    labels = [
        '深夜(00–05点)', 
        '早晨(06–08点)', 
        '上午(09–16点)', 
        '傍晚(17–20点)', 
        '晚间(21–23点)', 
        '其他时间段'      # 补齐缺失的第6个标签
    ]
    
    # include_lowest=True 确保最小值(如恰好为0)能被归入第一个区间
    df['time_period'] = pd.cut(df['hour'], bins=bins, labels=labels, right=False, include_lowest=True)

else:
    # 如果仅有年月日，则按星期几进行分析
    print("💡 提示: 当前'日期'列未包含具体时分秒，改为按【星期】分析活动规律:")
    df['weekday'] = df[real_time_col].dt.day_name()
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    cn_weekdays = {'Monday':'周一','Tuesday':'周二','Wednesday':'周三','Thursday':'周四','Friday':'周五','Saturday':'周六','Sunday':'周日'}
    
    avg_steps_by_day = df.groupby('weekday')[real_steps_col].mean().reindex(weekday_order).round(1)
    for day_en, steps in avg_steps_by_day.items():
        print(f"• {cn_weekdays[day_en]}: 平均步数 {steps} 步")

# ================= 3. 二、健康指标关注度分析 =================
print("\n【二、健康指标关注度分析】")

# 检查是否存在指标/场景列
metric_candidates = ['查看指标', '指标类型', '功能调用类型', '使用场景', '指标名称']
real_metric_col = next((col for col in metric_candidates if col in df.columns), None)

if real_metric_col:
    metric_counts = df[real_metric_col].value_counts()
    most_viewed = metric_counts.idxmax()
    least_viewed_list = metric_counts.tail(2).index.tolist()
    print(f"• 最常被查看指标: {most_viewed} (调用次数 {metric_counts.max()})")
    print(f"• 较少被查看指标: {', '.join(least_viewed_list)}")
else:
    print("⚠️ 当前数据集中未找到'查看指标'相关字段，跳过此部分分析。")

# ================= 4. 三、数据同步性能分析 =================
print("\n【三、数据同步性能分析】")

# 检查是否存在同步延迟列
sync_candidates = ['同步延迟', '响应时间', '延迟(ms)', '耗时', '同步时间']
real_sync_col = next((col for col in sync_candidates if col in df.columns), None)

if real_sync_col:
    df[real_sync_col] = pd.to_numeric(df[real_sync_col], errors='coerce')
    avg_sync_delay = df[real_sync_col].mean()
    print(f"• 平均同步延迟: 约 {avg_sync_delay:.2f} 秒")
    print("• 影响因素可能包括: 实时性强且采样数据量大；网络状态不佳或上传策略不合理。")
else:
    print("⚠️ 当前数据集中未找到'同步延迟'相关字段，跳过此部分分析。")


       用户                   日期  步数
0  User_1  2024-09-15 00:00:00   0
1  User_1  2024-09-15 01:00:00   0
2  User_1  2024-09-15 02:00:00   0
3  User_1  2024-09-15 03:00:00   0
4  User_1  2024-09-15 04:00:00   0
📊 当前数据集的真实列名如下:
   [0] 用户
   [1] 日期
   [2] 步数
✅ 成功识别到核心字段，开始分析...

【一、用户活动模式分析】

【二、健康指标关注度分析】
⚠️ 当前数据集中未找到'查看指标'相关字段，跳过此部分分析。

【三、数据同步性能分析】
⚠️ 当前数据集中未找到'同步延迟'相关字段，跳过此部分分析。
